<a href="https://colab.research.google.com/github/shribr/Bricky/blob/feat/dinov2-retrieval-prototype/Tools/dinov2-embeddings/dinov2_retrieval_prototype.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DINOv2 Minifigure Retrieval Prototype

Evaluate DINOv2 as a drop-in replacement for the shipped SimCLR torso encoder.

**Runtime:** Select **Runtime → Change runtime type → GPU** (free T4 is enough; A100 finishes the full catalog in ~3 min).

## 1. Clone repo & install dependencies

In [ ]:
!git clone https://github.com/shribr/Bricky.git
%cd Bricky

In [ ]:
!pip install -q -r Tools/dinov2-embeddings/requirements.txt

## 2. Build the evaluation set

Deterministic given the seed. Creates 200 figures × 8 variants = 1 600 query images.

In [ ]:
!python Tools/dinov2-embeddings/build_eval_set.py \
    --count 200 --variants-per-figure 8 --seed 42

## 3. Embed the catalog with DINOv2

Start with **ViT-S** (fastest, smallest bundle). If recall is too low, re-run with `dinov2_vitb14` or `dinov2_vitl14`.

In [ ]:
MODEL = "dinov2_vits14"  # Options: dinov2_vits14, dinov2_vitb14, dinov2_vitl14

In [ ]:
!python Tools/dinov2-embeddings/embed_catalog.py \
    --model {MODEL} \
    --out Tools/dinov2-embeddings/index/{MODEL}

## 4. Evaluate DINOv2 retrieval

In [ ]:
!python Tools/dinov2-embeddings/evaluate_retrieval.py \
    --index Tools/dinov2-embeddings/index/{MODEL} \
    --report Tools/dinov2-embeddings/reports/{MODEL}.json

## 5. Score the shipped SimCLR index (baseline)

Requires `Tools/torso-embeddings/out/torso_encoder.pt` to be present.

In [ ]:
!python Tools/dinov2-embeddings/compare_existing.py \
    --report Tools/dinov2-embeddings/reports/simclr_shipped.json

## 6. Compare reports

Load both JSON reports and compare `recall@1`, `recall@5`, `recall@10`.

| recall@5 | Interpretation |
|----------|----------------|
| < 0.20   | Encoder is effectively blind — likely a crop/preprocessing bug |
| 0.20–0.50 | Weak (expected for current SimCLR) |
| 0.50–0.75 | Usable as a top-8 candidate list |
| > 0.75   | Ship it |

In [ ]:
import json, pathlib

def load_report(path):
    with open(path) as f:
        return json.load(f)

dino_report = load_report(f"Tools/dinov2-embeddings/reports/{MODEL}.json")
simclr_path = pathlib.Path("Tools/dinov2-embeddings/reports/simclr_shipped.json")

print(f"=== DINOv2 ({MODEL}) ===")
for k in ["recall@1", "recall@5", "recall@10", "recall@50"]:
    print(f"  {k}: {dino_report.get(k, 'N/A')}")

if simclr_path.exists():
    simclr_report = load_report(simclr_path)
    print(f"\n=== SimCLR (shipped) ===")
    for k in ["recall@1", "recall@5", "recall@10", "recall@50"]:
        print(f"  {k}: {simclr_report.get(k, 'N/A')}")
else:
    print("\nSimCLR report not found — skipping comparison.")

## 7. Inspect failures

The `sample_failures` field lists the 20 worst cases. If the top-5 for a failed query is full of figures with the same torso color but different prints, the encoder is confusing "color" for "identity" — try a stronger backbone or fine-tuning. If the top-5 looks random, there's a preprocessing mismatch.

In [ ]:
failures = dino_report.get("sample_failures", [])
print(f"{len(failures)} sample failures:\n")
for i, f in enumerate(failures[:10], 1):
    print(f"{i}. {f}")

## Next steps

Once you have a winning model, swap the bundled assets:

```bash
cp Tools/dinov2-embeddings/index/<winner>/torso_embeddings.bin \
   Bricky/Resources/TorsoEmbeddings/torso_embeddings.bin
cp Tools/dinov2-embeddings/index/<winner>/torso_embeddings_index.json \
   Bricky/Resources/TorsoEmbeddings/torso_embeddings_index.json
```

Delete `torso_embeddings_mean.bin` — mean-centering isn't needed with DINOv2.

A matching `TorsoEncoder.mlmodel` (DINOv2 → CoreML conversion) is the one new piece needed after the offline numbers justify it.